# 04 — ANOVA Decomposition

Reproduces paper **Appendix Table 6** (η²_prompt, η²_question,
η²_interaction averaged across layers and concepts, per model).

Implementation in `grace/diagnostics/anova.py`. Reads activations
directly — no judging or steering needed.

> **Note:** This notebook requires the full raw activations (~122 GB) to
> recompute ANOVA from scratch. If activations are not available, it falls
> back to the pre-computed per-concept JSON summaries in `results/anova/`.
> To regenerate from scratch, run the full activation-extraction pipeline
> first (see README).

In [ ]:
from pathlib import Path
import json
import pandas as pd

from grace.diagnostics.anova import anova_decomposition

MODELS = {
    'gemma2-2b': 'google/gemma-2-2b-it',
    'gemma3-27b': 'google/gemma-3-27b-it',
    'llama3-70b': 'meta-llama/Llama-3.3-70B-Instruct',
}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))
OUT = Path('results/anova'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
rows = []
for tag, mname in MODELS.items():
    model_dir = mname.split('/')[-1]
    for c in CONCEPTS:
        # Try live computation from activations first; fall back to cached JSON.
        cached = OUT / model_dir / f'{c}.json'
        try:
            d = anova_decomposition(mname, c, activation_type='response_avg')
        except FileNotFoundError:
            if cached.exists():
                with open(cached) as f:
                    per_layer = json.load(f)['per_layer']
                vals = list(per_layer.values())
                d = {
                    'mean_eta_sq_prompt': sum(v['eta_sq_prompt'] for v in vals) / len(vals),
                    'mean_eta_sq_question': sum(v['eta_sq_question'] for v in vals) / len(vals),
                    'mean_eta_sq_interaction': sum(v['eta_sq_interaction'] for v in vals) / len(vals),
                }
            else:
                continue
        rows.append({
            'model': tag, 'concept': c,
            'eta_prompt': d['mean_eta_sq_prompt'],
            'eta_question': d['mean_eta_sq_question'],
            'eta_interaction': d['mean_eta_sq_interaction'],
        })
df = pd.DataFrame(rows)
summary = df.groupby('model')[['eta_prompt', 'eta_question', 'eta_interaction']].mean().round(3)
summary.loc['All models'] = summary.mean()
summary